# Donut — OCR-Free Invoice Field Extraction

This notebook fine-tunes the pretrained **Donut** model (`naver-clova-ix/donut-base`) on invoice images and uses it to extract structured fields directly — no OCR step involved.

**Pipeline:**
```
invoice image → Donut (vision encoder + text decoder) → JSON output
```

**Target fields:**
- `invoice_number`
- `date`
- `total`

---
## 1. Setup

In [2]:
import os
import json
import re
import random
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DonutProcessor,
    VisionEncoderDecoderModel,
    VisionEncoderDecoderConfig,
)
from transformers import get_scheduler
from torch.optim import AdamW
from tqdm.auto import tqdm

# ── Configuration ──────────────────────────────────────────────────────────────
MODEL_NAME = "naver-clova-ix/donut-base"

# Robust project path resolution (works from repo root or Donut folder).
_cwd = Path.cwd()
candidate_roots = [
    _cwd,
    _cwd.parent,
    Path("/Users/joudtaher/statistical_learning_group_assignment"),
]
PROJECT_ROOT = next(
    (p for p in candidate_roots if (p / "ai" / "extraction" / "Donut").exists()),
    Path("/Users/joudtaher/statistical_learning_group_assignment"),
)

DONUT_DIR = PROJECT_ROOT / "ai" / "extraction" / "Donut"
DATA_DIR = DONUT_DIR / "data"
IMAGE_DIR = DATA_DIR / "images"
LABELS_TRAIN_FILE = DATA_DIR / "labels_train.json"
LABELS_VAL_FILE = DATA_DIR / "labels_val.json"
LABELS_FILE = DATA_DIR / "labels.json"
LEGACY_LABEL_FILE = DATA_DIR / "label.json"  # optional fallback if present
if not LABELS_FILE.exists() and LEGACY_LABEL_FILE.exists():
    LABELS_FILE = LEGACY_LABEL_FILE

OUTPUT_DIR = DONUT_DIR / "outputs" / "donut_finetuned"

MAX_LENGTH = 192     # max decoder token length
IMAGE_SIZE = [1280, 960]  # (height, width) — Donut default
BATCH_SIZE = 2
NUM_EPOCHS = 5
LR = 5e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Expanded schema for Donut extraction.
TARGET_FIELDS = [
    "invoice_number",
    "invoice_date",
    "due_date",
    "issuer_name",
    "client_name",
    "client_email",
    "client_phone",
    "billing_address",
    "shipping_address",
    "subtotal",
    "vat",
    "total",
    "discount",
    "vat_rate",
    "discount_rate",
]

# Task prompt that tells the decoder what to produce
TASK_PROMPT = "<s_invoice>"

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"CWD: {_cwd}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Donut dir: {DONUT_DIR}")
print(f"Data dir: {DATA_DIR}")
print(f"Images dir: {IMAGE_DIR}")
print(f"Train labels: {LABELS_TRAIN_FILE}")
print(f"Val labels: {LABELS_VAL_FILE}")
print(f"All labels: {LABELS_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Target fields ({len(TARGET_FIELDS)}): {', '.join(TARGET_FIELDS)}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Device: cpu
PyTorch: 2.11.0
CWD: /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut
Project root: /Users/joudtaher/statistical_learning_group_assignment
Donut dir: /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut
Data dir: /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/data
Images dir: /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/data/images
Train labels: /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/data/labels_train.json
Val labels: /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/data/labels_val.json
All labels: /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/data/labels.json
Output dir: /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/outputs/donut_finetuned
Target fields (15): invoice_number, invoice_date, due_date, issuer_name, client_name, client_email, client_phone,

---
## 2. Data Preparation

This notebook uses the exact project paths below:
```text
ai/extraction/Donut/data/images
ai/extraction/Donut/data/labels_train.json
ai/extraction/Donut/data/labels_val.json
ai/extraction/Donut/data/labels.json
```

Generate the Donut dataset with OCR bootstrapped labels:
```bash
PYTHONPATH="/Users/joudtaher/statistical_learning_group_assignment" /Users/joudtaher/statistical_learning_group_assignment/.venv/bin/python /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/prepare_donut_data.py --label-source ocr --clean
```

Expected files:
```text
data/
  images/
    train/
    val/
  labels_train.json
  labels_val.json
  labels.json
```

`labels_train.json` and `labels_val.json` are used directly throughout training/evaluation to avoid split drift.

In [2]:
# ── Data file sanity checks (source of truth) ────────────────────────────────
required_setup_vars = [
    "IMAGE_DIR",
    "LABELS_TRAIN_FILE",
    "LABELS_VAL_FILE",
    "LABELS_FILE",
    "PROJECT_ROOT",
    "DATA_DIR",
]
missing_setup = [name for name in required_setup_vars if name not in globals()]
if missing_setup:
    raise RuntimeError(
        "Setup variables are missing. Run Cell 3 (Setup) first, then rerun this cell. "
        f"Missing: {', '.join(missing_setup)}"
    )

required_paths = [
    IMAGE_DIR / "train",
    IMAGE_DIR / "val",
    LABELS_TRAIN_FILE,
    LABELS_VAL_FILE,
    LABELS_FILE,
]

missing = [p for p in required_paths if not p.exists()]
if missing:
    print(f"CWD: {Path.cwd()}")
    print(f"Project root (from setup): {PROJECT_ROOT}")
    print("Missing Donut data files/folders:")
    for path in missing:
        print(f"  - {path}")
    raise FileNotFoundError(
        "Run: PYTHONPATH=\"/Users/joudtaher/statistical_learning_group_assignment\" "
        "/Users/joudtaher/statistical_learning_group_assignment/.venv/bin/python "
        "/Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/prepare_donut_data.py "
        "--label-source ocr --clean"
    )

# Guardrail: stop early if labels are empty placeholders.
with open(LABELS_TRAIN_FILE) as f_train:
    train_preview = json.load(f_train)
with open(LABELS_VAL_FILE) as f_val:
    val_preview = json.load(f_val)

def _is_empty_record(rec: dict) -> bool:
    try:
        gt = json.loads(rec.get("ground_truth", "{}"))
    except json.JSONDecodeError:
        return False
    if not isinstance(gt, dict):
        return True
    return all(not str(value).strip() for value in gt.values())

preview = train_preview[:30] + val_preview[:30]
if preview and all(_is_empty_record(r) for r in preview):
    raise ValueError(
        "Detected empty ground-truth labels. Regenerate data with OCR labels: "
        "python ai/extraction/Donut/prepare_donut_data.py --label-source ocr --clean"
    )

print("Donut data files are present and labels are non-empty.")
print(f"Train images dir: {IMAGE_DIR / 'train'}")
print(f"Val images dir:   {IMAGE_DIR / 'val'}")
print(f"Train labels:     {LABELS_TRAIN_FILE}")
print(f"Val labels:       {LABELS_VAL_FILE}")

Donut data files are present and labels are non-empty.
Train images dir: /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/data/images/train
Val images dir:   /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/data/images/val
Train labels:     /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/data/labels_train.json
Val labels:       /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/data/labels_val.json


In [3]:
# ── Dataset class ───────────────────────────────────────────────────────────

class InvoiceDonutDataset(Dataset):
    def __init__(self, records, image_dir, processor, max_length, task_prompt):
        self.records = records
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.max_length = max_length
        self.task_prompt = task_prompt

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        image = Image.open(self.image_dir / record["file"]).convert("RGB")

        # Build target string: <task_prompt><s>JSON</s>
        target_sequence = (
            self.task_prompt
            + self.processor.tokenizer.bos_token
            + record["ground_truth"]
            + self.processor.tokenizer.eos_token
        )

        pixel_values = self.processor(
            image, return_tensors="pt"
        ).pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            target_sequence,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)

        # Replace padding token id with -100 so it is ignored in loss
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}


def load_records() -> tuple[list[dict], list[dict]]:
    # Prefer explicit split files generated by prepare_donut_data.py
    if LABELS_TRAIN_FILE.exists() and LABELS_VAL_FILE.exists():
        with open(LABELS_TRAIN_FILE) as f_train:
            train = json.load(f_train)
        with open(LABELS_VAL_FILE) as f_val:
            val = json.load(f_val)
        return train, val

    # Fallback: deterministic split from labels.json
    with open(LABELS_FILE) as f_all:
        all_records = json.load(f_all)
    random.seed(42)
    random.shuffle(all_records)
    split = int(0.8 * len(all_records))
    return all_records[:split], all_records[split:]


train_records, val_records = load_records()
all_records = train_records + val_records
print(f"Train: {len(train_records)}  |  Val: {len(val_records)}")

Train: 480  |  Val: 119


---
## 3. Load Pretrained Donut

In [3]:
processor = DonutProcessor.from_pretrained(MODEL_NAME)
model     = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# ── Add task token to the tokenizer ─────────────────────────────────────────
new_special_tokens = [TASK_PROMPT]
processor.tokenizer.add_special_tokens({"additional_special_tokens": new_special_tokens})
model.decoder.resize_token_embeddings(len(processor.tokenizer), mean_resizing=False)

# ── Align processor image size with model config ─────────────────────────────
img_proc = processor.image_processor if hasattr(processor, "image_processor") else processor.feature_extractor
img_proc.size = {"height": IMAGE_SIZE[0], "width": IMAGE_SIZE[1]}
if hasattr(img_proc, "do_align_long_axis"):
    img_proc.do_align_long_axis = False

model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(TASK_PROMPT)
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.eos_token_id

# Keep generation params in generation_config (Transformers >= 4.4x requirement)
model.generation_config.max_length = MAX_LENGTH

model.to(DEVICE)
print("Model loaded.")

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

Model loaded.


---
## 4. Fine-Tuning

In [5]:
train_dataset = InvoiceDonutDataset(train_records, IMAGE_DIR, processor, MAX_LENGTH, TASK_PROMPT)
val_dataset   = InvoiceDonutDataset(val_records,   IMAGE_DIR, processor, MAX_LENGTH, TASK_PROMPT)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=NUM_EPOCHS * len(train_loader),
)

best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]"):
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels       = batch["labels"].to(DEVICE)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    avg_train = train_loss / len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [val]"):
            pixel_values = batch["pixel_values"].to(DEVICE)
            labels       = batch["labels"].to(DEVICE)
            outputs      = model(pixel_values=pixel_values, labels=labels)
            val_loss    += outputs.loss.item()

    avg_val = val_loss / len(val_loader)
    print(f"Epoch {epoch} — train loss: {avg_train:.4f}  |  val loss: {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)
        print(f"  ✓ Saved best model (val loss: {best_val_loss:.4f})")

print("Fine-tuning complete.")

Epoch 1/5 [train]:   0%|          | 0/240 [00:00<?, ?it/s]

Epoch 1/5 [val]:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 1 — train loss: 1.5304  |  val loss: 0.1854


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Saved best model (val loss: 0.1854)


...

Epoch 4/5 [val]:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 4 — train loss: 0.0439  |  val loss: 0.1282


Epoch 5/5 [train]:   0%|          | 0/240 [00:00<?, ?it/s]

Epoch 5/5 [val]:   0%|          | 0/60 [00:00<?, ?it/s]

Epoch 5 — train loss: 0.0223  |  val loss: 0.1339
Fine-tuning complete.


---
## 5. Inference on a Single Invoice Image

In [5]:
def extract_invoice_fields(image_path: str, checkpoint_dir: str = str(OUTPUT_DIR)) -> dict:
    """Run fine-tuned Donut on a single invoice image and return extracted fields."""
    proc  = DonutProcessor.from_pretrained(checkpoint_dir)
    mdl   = VisionEncoderDecoderModel.from_pretrained(checkpoint_dir).to(DEVICE)
    mdl.eval()

    image = Image.open(image_path).convert("RGB")
    pixel_values = proc(image, return_tensors="pt").pixel_values.to(DEVICE)

    task_token_id = proc.tokenizer.convert_tokens_to_ids(TASK_PROMPT)
    decoder_input = torch.full((1, 1), task_token_id, dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        generated = mdl.generate(
            pixel_values,
            decoder_input_ids=decoder_input,
            max_length=MAX_LENGTH,
            pad_token_id=proc.tokenizer.pad_token_id,
            eos_token_id=proc.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=1,
            bad_words_ids=[[proc.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    sequence = proc.batch_decode(generated.sequences)[0]
    # Strip special tokens and task prompt
    sequence = sequence.replace(proc.tokenizer.eos_token, "").replace(proc.tokenizer.pad_token, "")
    sequence = re.sub(r"<.*?>", "", sequence).strip()

    try:
        result = json.loads(sequence)
    except json.JSONDecodeError:
        result = {"raw_output": sequence}

    return result


## ── Example ─────────────────────────────────────────────────────────────────
TEST_IMAGE = DONUT_DIR.parent / "image.png"  # ai/extraction/image.png

if not TEST_IMAGE.exists():
    raise FileNotFoundError(f"Test image not found: {TEST_IMAGE}")

result = extract_invoice_fields(str(TEST_IMAGE))
print(json.dumps(result, indent=2))

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

{
  "invoice_number": "",
  "invoice_date": "",
  "due_date": "",
  "issuer_name": "Founteinhead A+E",
  "client_name": "",
  "client_email": "",
  "client_phone": "",
  "billing_address": "",
  "shipping_address": "",
  "subtotal": "",
  "vat": "",
  "total": "10020",
  "discount": "",
  "vat_rate": "",
  "discount_rate": ""
}


---
## 6. Evaluation

We compute **exact match** and **partial match** (case-insensitive, stripped) for each target field across the validation set.

In [ ]:
def normalise(value: str) -> str:
    """Lowercase, strip whitespace and common currency symbols."""
    return re.sub(r"[\$€£,]", "", str(value)).strip().lower()


def evaluate_on_val(val_records, image_dir, checkpoint_dir):
    """Evaluate Donut with support-aware metrics.

    Exact/partial are computed only on rows where the ground truth for that field is non-empty,
    which avoids inflated scores from matching blank values.
    """
    n = len(val_records)
    if n == 0:
        print("Validation set is empty.")
        return

    stats = {
        field: {
            "support": 0,
            "pred_non_empty": 0,
            "exact_hits": 0,
            "partial_hits": 0,
        }
        for field in TARGET_FIELDS
    }

    for record in tqdm(val_records, desc="Evaluating (support-aware)"):
        gt = json.loads(record["ground_truth"])
.        pred = extract_invoice_fields(str(Path(image_dir) / record["file"]), checkpoint_dir)

        for field in TARGET_FIELDS:
            gt_val = normalise(gt.get(field, ""))
            pred_val = normalise(pred.get(field, ""))

            if pred_val:
                stats[field]["pred_non_empty"] += 1

            if not gt_val:
                continue

            stats[field]["support"] += 1

            if gt_val == pred_val:
                stats[field]["exact_hits"] += 1

            # Symmetric partial check to capture close-but-not-identical strings.
            if pred_val and (gt_val in pred_val or pred_val in gt_val):
                stats[field]["partial_hits"] += 1

    print(f"\nSupport-aware results over {n} validation samples:\n")
    print(
        f"{'Field':<20} {'Support':>8} {'Pred!=Empty':>12} {'Exact@Support':>14} {'Partial@Support':>16}"
    )
    print("-" * 76)

    for field in TARGET_FIELDS:
        support = stats[field]["support"]
        pred_non_empty = stats[field]["pred_non_empty"]
        exact_hits = stats[field]["exact_hits"]
        partial_hits = stats[field]["partial_hits"]

        exact_pct = (exact_hits / support * 100) if support else 0.0
        partial_pct = (partial_hits / support * 100) if support else 0.0
        pred_non_empty_pct = pred_non_empty / n * 100

        print(
            f"{field:<20} {support:>8} {pred_non_empty_pct:>11.1f}%"
            f" {exact_pct:>13.1f}% {partial_pct:>15.1f}%"
        )


evaluate_on_val(val_records, IMAGE_DIR, str(OUTPUT_DIR))

IndentationError: unindent does not match any outer indentation level (<string>, line 50)

---
## 7. YOLO Benchmark

We compare Donut's field-level accuracy against the best YOLO detection checkpoint.
YOLO works at the **bounding-box** level — it predicts *where* a field is, not *what* it says.
We therefore measure **field detection rate**: the fraction of validation images on which YOLO
returns at least one box for each target field class (`Numéro_Facture`, `Date_Facturation`,
`Total_TTC`), using a confidence threshold of 0.25.

This gives a fair apples-to-apples comparison: Donut exact/partial match vs YOLO detection rate.

In [13]:
# ── YOLO Benchmark ───────────────────────────────────────────────────────────
# Requires: pip install ultralytics

from pathlib import Path

# Best YOLO checkpoint trained on the same invoice dataset
YOLO_CHECKPOINT = (
    DONUT_DIR.parent / "YOLO_method" / "runs" / "invoice_fields_960_best2" / "weights" / "best.pt"
)
YOLO_CONF = 0.25   # confidence threshold

# Mapping from YOLO class name → Donut field key
YOLO_CLASS_TO_FIELD = {
    "Numéro_Facture": "invoice_number",
    "Date_Facturation": "date",
    "Total_TTC": "total",
}


def yolo_field_detection_rate(val_records, image_dir, checkpoint, conf=0.25):
    """Return per-field detection rates for YOLO on the validation set."""
    try:
        from ultralytics import YOLO
    except ImportError:
        raise ImportError("Install ultralytics: pip install ultralytics")

    if not Path(checkpoint).exists():
        raise FileNotFoundError(f"YOLO checkpoint not found: {checkpoint}")

    yolo = YOLO(str(checkpoint))

    detected = {f: 0 for f in YOLO_CLASS_TO_FIELD.values()}
    n = len(val_records)

    for record in tqdm(val_records, desc="YOLO benchmark"):
        img_path = str(Path(image_dir) / record["file"])
        results = yolo.predict(source=img_path, conf=conf, verbose=False)

        found_classes = set()
        for r in results:
            for cls_id in r.boxes.cls.int().tolist():
                cls_name = r.names[cls_id]
                if cls_name in YOLO_CLASS_TO_FIELD:
                    found_classes.add(YOLO_CLASS_TO_FIELD[cls_name])

        for field in detected:
            if field in found_classes:
                detected[field] += 1

    return {field: detected[field] / n * 100 for field in detected}, n


def benchmark_comparison(val_records, image_dir, donut_checkpoint, yolo_checkpoint):
    """Print a side-by-side table of Donut vs YOLO accuracy metrics."""
    # ── Donut metrics ────────────────────────────────────────────────────────
    proc = DonutProcessor.from_pretrained(donut_checkpoint)
    mdl = VisionEncoderDecoderModel.from_pretrained(donut_checkpoint).to(DEVICE)
    mdl.eval()

    exact_hits = {"invoice_number": 0, "date": 0, "total": 0}
    partial_hits = {"invoice_number": 0, "date": 0, "total": 0}
    n = len(val_records)

    for record in tqdm(val_records, desc="Donut evaluation"):
        gt = json.loads(record["ground_truth"])
        pred = extract_invoice_fields(str(Path(image_dir) / record["file"]), donut_checkpoint)
        for field in exact_hits:
            gt_v = normalise(gt.get(field, ""))
            pred_v = normalise(pred.get(field, ""))
            if gt_v == pred_v:
                exact_hits[field] += 1
            if gt_v and gt_v in pred_v:
                partial_hits[field] += 1

    donut_exact = {f: exact_hits[f] / n * 100 for f in exact_hits}
    donut_partial = {f: partial_hits[f] / n * 100 for f in partial_hits}

    # ── YOLO metrics ─────────────────────────────────────────────────────────
    yolo_detect, _ = yolo_field_detection_rate(val_records, image_dir, yolo_checkpoint)

    # ── Print table ──────────────────────────────────────────────────────────
    print(f"\nBenchmark — {n} validation images\n")
    header = f"{'Field':<20} {'Donut Exact':>12} {'Donut Partial':>14} {'YOLO Detected':>14}"
    print(header)
    print("-" * len(header))
    for field in ("invoice_number", "date", "total"):
        print(
            f"{field:<20} {donut_exact[field]:>11.1f}%"
            f"  {donut_partial[field]:>12.1f}%"
            f"  {yolo_detect[field]:>12.1f}%"
        )


benchmark_comparison(
    val_records,
    IMAGE_DIR,
    donut_checkpoint=str(OUTPUT_DIR),
    yolo_checkpoint=YOLO_CHECKPOINT,
)


Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Donut evaluation:   0%|          | 0/119 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

YOLO benchmark:   0%|          | 0/119 [00:00<?, ?it/s]


Benchmark — 119 validation images

Field                 Donut Exact  Donut Partial  YOLO Detected
---------------------------------------------------------------
invoice_number              88.2%           6.7%          71.4%
date                       100.0%           0.0%          71.4%
total                       43.7%          40.3%          84.0%


In [6]:
# ── Extra single-image test (new invoice) ─────────────────────────────────────
NEW_TEST_IMAGE = DONUT_DIR / "test_img.jpeg"

if not NEW_TEST_IMAGE.exists():
    raise FileNotFoundError(
        f"New test image not found: {NEW_TEST_IMAGE}\n"
        "Update NEW_TEST_IMAGE to your added file path, then rerun this cell."
    )

new_result = extract_invoice_fields(str(NEW_TEST_IMAGE), checkpoint_dir=str(OUTPUT_DIR))
print(f"Inference image: {NEW_TEST_IMAGE}")
print(json.dumps(new_result, indent=2, ensure_ascii=False))

Loading weights:   0%|          | 0/483 [00:00<?, ?it/s]

Inference image: /Users/joudtaher/statistical_learning_group_assignment/ai/extraction/Donut/test_img.jpeg
{
  "invoice_number": "",
  "invoice_date": "",
  "due_date": "",
  "issuer_name": "LAuxuryStore",
  "client_name": "",
  "client_email": "",
  "client_phone": "",
  "billing_address": "",
  "shipping_address": "",
  "subtotal": "",
  "vat": "",
  "total": "49628-7978",
  "discount": "",
  "vat_rate": "",
  "discount_rate": ""
}
